# T9.3 Tech News Tracker — Offline Evaluation

**Goal:** Measure how well `facebook/bart-large-mnli` classifies tech vs non-tech headlines  
**Dataset:** [India Headlines News Dataset](https://www.kaggle.com/datasets/therohk/india-headlines-news-dataset) — 21 years of labelled Indian headlines  
**Task:** Binary evaluation (tech / non-tech) + subcategory confusion matrix

---
### Setup
Download the dataset from Kaggle first:
```bash
pip install kaggle
kaggle datasets download -d therohk/india-headlines-news-dataset
unzip india-headlines-news-dataset.zip
```
Or place `india-headlines-news-dataset.csv` in the `notebooks/` folder manually.

In [ ]:
# Install dependencies (run once on Colab)
# !pip install transformers torch accelerate pandas scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print(f'Torch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')

## 1. Load the India Headlines Dataset

In [ ]:
CSV_PATH = 'india-news-headlines.csv'

df = pd.read_csv(CSV_PATH)
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head(3)

In [ ]:
print('Category counts:')
print(df['headline_category'].value_counts())

## 2. Build a Balanced Evaluation Sample

Binary classification task: `tech` vs `non-tech`.  
Ground truth comes from the dataset's existing `headline_category` labels.

In [ ]:
SAMPLE_PER_CLASS = 150
RANDOM_SEED = 42

tech_df = (
    df[df['headline_category'].str.lower() == 'tech']
    .dropna(subset=['headline_text'])
    .sample(n=SAMPLE_PER_CLASS, random_state=RANDOM_SEED)
    .assign(true_label='tech')
)

non_tech_cats = ['entertainment', 'sports', 'lifestyle', 'world']
non_tech_df = (
    df[df['headline_category'].str.lower().isin(non_tech_cats)]
    .dropna(subset=['headline_text'])
    .sample(n=SAMPLE_PER_CLASS, random_state=RANDOM_SEED)
    .assign(true_label='non-tech')
)

eval_df = pd.concat([tech_df, non_tech_df]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
print(f'Evaluation set: {len(eval_df)} headlines')
print(eval_df['true_label'].value_counts())
eval_df[['headline_text', 'headline_category', 'true_label']].head(5)

## 3. Load the Zero-Shot Classifier

In [ ]:
device = 0 if torch.cuda.is_available() else -1
print(f'Using: {"GPU" if device == 0 else "CPU"}')

clf = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=device,
)
print('Model loaded.')

## 4. Binary Evaluation — Tech vs Non-Tech

In [ ]:
BINARY_LABELS = ['technology news', 'non-technology news']

texts = eval_df['headline_text'].tolist()
results = clf(texts, candidate_labels=BINARY_LABELS, batch_size=16, multi_label=False)

LABEL_MAP = {'technology news': 'tech', 'non-technology news': 'non-tech'}
eval_df['pred_label'] = [LABEL_MAP[r['labels'][0]] for r in results]
eval_df['confidence'] = [r['scores'][0] for r in results]

eval_df[['headline_text', 'true_label', 'pred_label', 'confidence']].head(10)

## 5. Metrics

In [ ]:
y_true = eval_df['true_label'].tolist()
y_pred = eval_df['pred_label'].tolist()

acc = accuracy_score(y_true, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label='tech')

print(f'Accuracy  : {acc:.3f}')
print(f'Precision : {prec:.3f}')
print(f'Recall    : {rec:.3f}')
print(f'F1 Score  : {f1:.3f}')
print()
print(classification_report(y_true, y_pred, target_names=['non-tech', 'tech']))

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=['tech', 'non-tech'])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Tech', 'Non-Tech'])

fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — Tech vs Non-Tech\n(facebook/bart-large-mnli, zero-shot)', fontsize=11)
plt.tight_layout()
plt.savefig('../report/confusion_matrix.png', dpi=150)
plt.show()

## 6. Confidence Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

correct = eval_df[eval_df['true_label'] == eval_df['pred_label']]['confidence']
wrong   = eval_df[eval_df['true_label'] != eval_df['pred_label']]['confidence']

ax.hist(correct, bins=20, alpha=0.7, label='Correct', color='steelblue')
ax.hist(wrong,   bins=20, alpha=0.7, label='Wrong',   color='salmon')
ax.set_xlabel('Top-label confidence')
ax.set_ylabel('Count')
ax.set_title('Confidence Distribution — Correct vs Wrong Predictions')
ax.legend()
plt.tight_layout()
plt.savefig('../report/confidence_dist.png', dpi=150)
plt.show()

## 7. Subcategory Distribution

In [ ]:
# Descriptive hypothesis labels — same as classifier.py in the app
HYPOTHESIS_LABELS = [
    'artificial intelligence or machine learning',
    'technology startups, funding, or business strategy',
    'consumer electronics, gadgets, chips, or hardware devices',
    'software applications, platforms, or developer tools',
    'general technology news',
]
DISPLAY_NAMES = [
    'AI & Machine Learning',
    'Startups & Business',
    'Gadgets & Hardware',
    'Software & Apps',
    'General Tech',
]
HYP_TO_DISPLAY = dict(zip(HYPOTHESIS_LABELS, DISPLAY_NAMES))

tech_headlines_df = eval_df[eval_df['true_label'] == 'tech'].copy()
tech_texts = tech_headlines_df['headline_text'].tolist()

sub_results = clf(tech_texts, candidate_labels=HYPOTHESIS_LABELS, batch_size=16, multi_label=False)
tech_headlines_df['model_label'] = [HYP_TO_DISPLAY[r['labels'][0]] for r in sub_results]

print('Model prediction distribution:')
print(tech_headlines_df['model_label'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = tech_headlines_df['model_label'].value_counts()
ax.barh(counts.index, counts.values, color='steelblue')
ax.set_xlabel('Number of headlines')
ax.set_title('Subcategory Distribution (zero-shot) on India Tech Headlines')
for i, v in enumerate(counts.values):
    ax.text(v + 0.5, i, str(v), va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../report/subcategory_dist.png', dpi=150)
plt.show()

## 8. Subcategory Confusion Matrix

The India Headlines Dataset has no fine-grained subcategory labels, so we build  
**keyword-based pseudo-labels** as an approximate reference.  
The matrix shows where the model's predictions agree or disagree with keyword-based intuition —  
it is a diagnostic tool, not a strict accuracy benchmark.

In [ ]:
# Keyword sets for each subcategory — ordered from most specific to least
KEYWORD_MAP = {
    'AI & Machine Learning': [
        'artificial intelligence', 'machine learning', 'deep learning',
        'neural network', 'gpt', 'llm', 'chatbot', 'chatgpt', 'openai',
        'robot', 'automation', ' ai ', 'generative',
    ],
    'Startups & Business': [
        'startup', 'funding', 'series a', 'series b', 'series c',
        'ipo', 'acquisition', 'unicorn', 'valuation', 'venture capital',
        'raises', 'merger', 'investment round',
    ],
    'Gadgets & Hardware': [
        'iphone', 'samsung', 'laptop', 'tablet', 'chip', 'processor',
        'gpu', 'smartphone', 'headphone', 'smartwatch', 'gaming console',
        'hardware', 'gadget', 'wearable', 'camera sensor',
    ],
    'Software & Apps': [
        'app ', 'software', 'update', 'platform', 'browser',
        'windows', 'android', 'ios ', 'google', 'microsoft',
        'apple', 'meta ', 'twitter', 'instagram', 'app store',
    ],
}

def keyword_label(headline: str) -> str:
    h = headline.lower()
    for cat, keywords in KEYWORD_MAP.items():
        if any(k in h for k in keywords):
            return cat
    return 'General Tech'

tech_headlines_df['keyword_label'] = tech_headlines_df['headline_text'].apply(keyword_label)

print('Keyword pseudo-label distribution:')
print(tech_headlines_df['keyword_label'].value_counts())
print()
print('Model prediction distribution:')
print(tech_headlines_df['model_label'].value_counts())

In [ ]:
sub_true = tech_headlines_df['keyword_label'].tolist()
sub_pred = tech_headlines_df['model_label'].tolist()

cm_sub = confusion_matrix(sub_true, sub_pred, labels=DISPLAY_NAMES)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_sub,
    annot=True, fmt='d', cmap='Blues',
    xticklabels=DISPLAY_NAMES,
    yticklabels=DISPLAY_NAMES,
    ax=ax,
)
ax.set_xlabel('Model Prediction', fontsize=12)
ax.set_ylabel('Keyword Pseudo-Label (approx. ground truth)', fontsize=12)
ax.set_title(
    'Subcategory Confusion Matrix\n'
    '(keyword pseudo-labels vs bart-large-mnli predictions)',
    fontsize=13,
)
plt.xticks(rotation=30, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../report/subcategory_confusion_matrix.png', dpi=150)
plt.show()
print('Saved to report/subcategory_confusion_matrix.png')

In [ ]:
print(classification_report(sub_true, sub_pred, labels=DISPLAY_NAMES, zero_division=0))

## 9. Ablation — Hypothesis Label Phrasing

MNLI performance is sensitive to how candidate labels are phrased.  
We compare three label sets for the binary task.

In [ ]:
label_sets = {
    'Descriptive':  ['technology news', 'non-technology news'],
    'Short':        ['tech', 'not tech'],
    'Hypothesis':   ['This headline is about technology.', 'This headline is not about technology.'],
}

ablation_rows = []
for name, labels in label_sets.items():
    preds_raw = clf(texts, candidate_labels=labels, batch_size=16, multi_label=False)
    preds = ['tech' if r['labels'][0] == labels[0] else 'non-tech' for r in preds_raw]
    acc_abl = accuracy_score(y_true, preds)
    _, _, f1_abl, _ = precision_recall_fscore_support(y_true, preds, average='binary', pos_label='tech')
    ablation_rows.append({'Label set': name, 'Accuracy': round(acc_abl, 3), 'F1': round(f1_abl, 3)})
    print(f'{name:15s} → Accuracy: {acc_abl:.3f}  F1: {f1_abl:.3f}')

pd.DataFrame(ablation_rows)

## Summary

| Metric | Value |
| --- | --- |
| Binary Accuracy | _fill after run_ |
| Binary F1 | _fill after run_ |
| Best label phrasing | _fill after ablation_ |
| Most confused subcategory pair | _fill after subcategory CM_ |

Plots saved to `report/` for inclusion in the PDF report:
- `confusion_matrix.png`
- `confidence_dist.png`
- `subcategory_dist.png`
- `subcategory_confusion_matrix.png`